# 模型评估与推理


## 1. 安装依赖


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "transformers==4.52.4", "sentencepiece>=0.2", "safetensors>=0.4"])


## 2. 导入与路径配置


In [ ]:
import csv
import json
from pathlib import Path

import torch
from google.colab import drive
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoConfig, AutoModel, AutoTokenizer

drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/prepared_data")
OUTPUT_DIR = Path("/content/drive/MyDrive/contrastive_outputs")
TASK = "C"  # A、B、C
SEED = 42
CHECKPOINT = OUTPUT_DIR / "model_runs" / f"{TASK}_scl_seed{SEED}" / "best"
EVALUATION_SPLIT = "grouped_multilingual"  # persian_external 为波斯语开发集
DEVICE = "cuda"
BATCH_SIZE = 8

IDENTITIES = ['l', 'g', 'b', 't', 'q', 'i', 'a', 'nb', 'lgbtqia+']
HATE = ['no', 'yes_implicit', 'yes_explicit']


## 3. 模型与数据读取


### 3.1 输入编码与批次


In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


In [ ]:
def encode_record(row, tokenizer, config):
    text = f"[COMMENT] {row['comment']} [TITLE] {row['title']} [DESCRIPTION] {row['description']}"
    return tokenizer(text, truncation=True, max_length=config['max_length'])


In [ ]:
class TextDataset(Dataset):
    def __init__(self, rows, tokenizer, config):
        self.rows = rows
        self.inputs = [encode_record(row, tokenizer, config) for row in rows]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return {'inputs': self.inputs[index], 'row': self.rows[index]}


In [ ]:
class PredictionCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, items):
        return {
            'inputs': self.tokenizer.pad([item['inputs'] for item in items], padding=True, return_tensors='pt'),
            'rows': [item['row'] for item in items],
        }


### 3.2 模型与检查点


In [ ]:
class TaskModel(nn.Module):
    def __init__(self, encoder, task, dropout=.1, projection_dim=128):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        sizes = {'A': {'a': 2}, 'B': {'b': 3}, 'C': {'scope': 2, 'identity': len(IDENTITIES)}}
        self.heads = nn.ModuleDict({name: nn.Linear(hidden, size) for name, size in sizes[task].items()})
        self.projection = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, projection_dim))

    def forward(self, **inputs):
        sequence = self.encoder(**inputs).last_hidden_state
        mask = inputs['attention_mask'].unsqueeze(-1).to(sequence.dtype)
        pooled = (sequence * mask).sum(1) / mask.sum(1).clamp_min(1)
        hidden = self.dropout(pooled)
        outputs = {name: head(hidden) for name, head in self.heads.items()}
        outputs['projection'] = F.normalize(self.projection(hidden), dim=-1)
        return outputs


In [ ]:
def load_checkpoint(root, device):
    config = read_json(root / 'config.json')
    encoder_config = AutoConfig.from_pretrained(root / 'encoder_config', local_files_only=True)
    encoder = AutoModel.from_config(encoder_config)
    model = TaskModel(encoder, config['task'], config['dropout'], config['projection_dim'])
    model.load_state_dict(torch.load(root / 'model.pt', map_location='cpu', weights_only=True))
    tokenizer = AutoTokenizer.from_pretrained(root / 'tokenizer', local_files_only=True)
    return model.to(device).eval(), tokenizer, config


### 3.3 评估数据


In [ ]:
def load_evaluation_rows(root, split_name, task):
    records = {}
    for path in sorted((root / 'processed').glob('*.jsonl')):
        for line in path.read_text(encoding='utf-8').splitlines():
            row = json.loads(line)
            records[row['id']] = row
    split = read_json(root / 'splits' / f'{split_name}.json')
    ids = split['evaluation_ids'] if split_name == 'persian_external' else split['validation_ids']
    rows = [records[sample_id] for sample_id in ids]
    if task == 'C':
        rows = [row for row in rows if row['target_loss_mask']]
    return rows


## 4. 预测与指标计算


### 4.1 解码与指标


In [ ]:
def classification_metrics(gold, pred, labels):
    scores = []
    for label in labels:
        tp = sum(g == label and p == label for g, p in zip(gold, pred))
        fp = sum(g != label and p == label for g, p in zip(gold, pred))
        fn = sum(g == label and p != label for g, p in zip(gold, pred))
        denominator = 2*tp+fp+fn
        scores.append({'label': label, 'support': sum(g == label for g in gold),
                       'precision': tp/(tp+fp) if tp+fp else 0.,
                       'recall': tp/(tp+fn) if tp+fn else 0.,
                       'f1': 2*tp/denominator if denominator else 0.})
    return {'n': len(gold), 'accuracy': sum(g == p for g, p in zip(gold, pred))/len(gold) if gold else None,
            'macro_f1': sum(s['f1'] for s in scores)/len(labels), 'per_class': scores}


In [ ]:
def decode(outputs, task, threshold=.5):
    if task == 'A':
        return [{'stereotype': 'yes' if label else 'no'} for label in outputs['a'].argmax(-1).tolist()]
    if task == 'B':
        return [{'hate_pred': label} for label in outputs['b'].argmax(-1).tolist()]
    scopes = outputs['scope'].argmax(-1).tolist()
    probabilities = outputs['identity'].sigmoid().tolist()
    predictions = []
    for scope, values in zip(scopes, probabilities):
        identities = [int(value >= threshold) for value in values]
        if not any(identities):
            identities[max(range(len(values)), key=values.__getitem__)] = 1
        target = ('group' if scope == 0 else 'individual') + '_' + ','.join(
            name for name, active in zip(IDENTITIES, identities) if active)
        predictions.append({'scope_pred': scope, 'identity_pred': identities, 'target': target})
    return predictions


In [ ]:
def evaluate_metrics(rows, predictions, task):
    if task == 'A':
        return classification_metrics([r['stereotype_label'] for r in rows],
                                      [int(p['stereotype'] == 'yes') for p in predictions], [0, 1])
    if task == 'B':
        return classification_metrics([r['hate_label'] for r in rows],
                                      [p['hate_pred'] for p in predictions], [0, 1, 2])
    scope = classification_metrics([r['scope_label'] for r in rows],
                                   [p['scope_pred'] for p in predictions], [0, 1])
    identities = []
    for i, name in enumerate(IDENTITIES):
        score = classification_metrics([r['identity_labels'][i] for r in rows],
                                       [p['identity_pred'][i] for p in predictions], [1])['per_class'][0]
        identities.append(dict(score, label=name))
    return {'n': len(rows), 'scope': scope, 'identity_per_label': identities,
            'identity_macro_f1': sum(score['f1'] for score in identities) / len(IDENTITIES),
            'target_exact_match': sum(r['target'] == p['target'] for r, p in zip(rows, predictions)) / len(rows)}


### 4.2 批量预测与结果保存


In [ ]:
@torch.no_grad()
def predict_rows(model, tokenizer, config, rows, device):
    dataset = TextDataset(rows, tokenizer, config)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, collate_fn=PredictionCollator(tokenizer))
    predictions = []
    for batch in loader:
        inputs = {name: value.to(device) for name, value in batch['inputs'].items()}
        outputs = model(**inputs)
        predictions.extend(decode(outputs, config['task'], config['identity_threshold']))
    return predictions


def predict_test_rows(model, tokenizer, config, rows, device, checkpoint):
    if config['task'] != 'C':
        return predict_rows(model, tokenizer, config, rows, device)

    b_checkpoint = checkpoint.parent.parent / f"B_scl_seed{config['seed']}" / 'best'
    b_model, b_tokenizer, b_config = load_checkpoint(b_checkpoint, device)
    hate_predictions = predict_rows(b_model, b_tokenizer, b_config, rows, device)
    del b_model

    hateful_indices = [i for i, prediction in enumerate(hate_predictions) if prediction['hate_pred'] != 0]
    hateful_rows = [rows[i] for i in hateful_indices]
    targets = predict_rows(model, tokenizer, config, hateful_rows, device)
    predictions = [{'target': 'none'} for _ in rows]
    for index, target in zip(hateful_indices, targets):
        predictions[index] = target
    return predictions


def write_predictions(path, rows, predictions, task):
    column = {'A': 'stereotype', 'B': 'hate_speech', 'C': 'target'}[task]
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['StereoQueerEval_id', column], delimiter='\t')
        writer.writeheader()
        for row, prediction in zip(rows, predictions):
            value = HATE[prediction['hate_pred']] if task == 'B' else prediction[column]
            writer.writerow({'StereoQueerEval_id': row['id'], column: value})


## 5. 评估最佳模型

C 仅评估真实仇恨样本，与 02 保持一致。


In [ ]:
device = torch.device(DEVICE)
model, tokenizer, config = load_checkpoint(CHECKPOINT, device)
rows = load_evaluation_rows(DATA_DIR, EVALUATION_SPLIT, config['task'])
predictions = predict_rows(model, tokenizer, config, rows, device)
metrics = evaluate_metrics(rows, predictions, config['task'])

result_dir = OUTPUT_DIR / 'evaluations' / CHECKPOINT.parent.name / EVALUATION_SPLIT
result_dir.mkdir(parents=True, exist_ok=True)
(result_dir / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
write_predictions(result_dir / 'predictions.tsv', rows, predictions, config['task'])

if config['task'] == 'C':
    print(f"样本：{len(rows)}，对象完全匹配率：{metrics['target_exact_match']:.6f}，"
          f"范围 Macro-F1：{metrics['scope']['macro_f1']:.6f}，身份 Macro-F1：{metrics['identity_macro_f1']:.6f}")
else:
    print(f"样本：{len(rows)}，Macro-F1：{metrics['macro_f1']:.6f}，准确率：{metrics['accuracy']:.6f}")
print("评估结果目录：", result_dir)


## 6. 无标签文件推理

C 自动加载同种子的 B 模型：非仇恨输出 `none`，仇恨样本再预测对象。


In [ ]:
RUN_PREDICTION = False
INPUT_PATH = Path("/content/drive/MyDrive/test.tsv")
PREDICTION_PATH = OUTPUT_DIR / 'predictions' / CHECKPOINT.parent.name / 'test.tsv'

if RUN_PREDICTION:
    with INPUT_PATH.open(encoding='utf-8-sig', newline='') as handle:
        test_rows = [
            {'id': row['StereoQueerEval_id'], 'comment': row['yt_comment'],
             'title': row['yt_title'], 'description': row['yt_description']}
            for row in csv.DictReader(handle, delimiter='\t')
        ]
    test_predictions = predict_test_rows(model, tokenizer, config, test_rows, device, CHECKPOINT)
    write_predictions(PREDICTION_PATH, test_rows, test_predictions, config['task'])
    print("预测文件：", PREDICTION_PATH)
